<a href="https://colab.research.google.com/github/rxnu/LLM-Project/blob/main/3_pre_trained_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3. Pre-trained Transformer Baseline

**Goals**  
1. Establish a **zero-shot** baseline by running the off-the-shelf SST-2 model on IMDB.  
2. Fine-tune `distilbert-base-uncased` on IMDB for a true baseline.  
3. Compare metrics


In [ ]:
# Install and Imports
!pip install -q transformers datasets evaluate torch

from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets    import load_dataset, DatasetDict
import evaluate, torch, numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 106.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 825.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 95.0 MB/s eta 0:00:00


In [ ]:
# Instantiate the SST-2 fine-tuned DistilBERT pipeline
pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
)

# Sample a small batch from IMDB test
ds = load_dataset("imdb", split="test").shuffle(seed=42).select(range(500))
texts = ds["text"][:8]

# Run predictions
print("SST-2 on IMDB Sample")
for txt, out in zip(texts, pipe(texts)):
    print(f"> {txt[:60]!r}… → {out['label']} ({out['score']:.3f})")


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cuda:0


SST-2 on IMDB Sample
> '<br /><br />When I unsuspectedly rented A Thousand Acres, I '… → POSITIVE (0.999)
> 'This is the latest entry in the long series of films with th'… → POSITIVE (0.997)
> 'This movie was so frustrating. Everything seemed energetic a'… → NEGATIVE (0.997)
> 'I was truly and wonderfully surprised at "O\' Brother, Where '… → NEGATIVE (0.649)
> 'This movie spends most of its time preaching that it is the '… → NEGATIVE (0.999)
> 'After a very long time Marathi cinema has come with some goo'… → POSITIVE (1.000)
> 'This is a really sad, and touching movie! It deals with the '… → POSITIVE (0.997)
> "Don't pay any attention to the rave reviews of this film her"… → NEGATIVE (1.000)


In [ ]:
# Load IMDB and create 22.5K/2.5K/25K splits
raw = load_dataset("imdb")
ds = DatasetDict({
    "train":      raw["train"].shuffle(42).select(range(22500)),
    "validation": raw["train"].shuffle(42).select(range(22500,25000)),
    "test":       raw["test"]
})

# Tokenizer for fine-tuning
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)

# Map + format for PyTorch
ds = ds.map(tokenize, batched=True)
ds = ds.remove_columns("text")
ds.set_format("torch", columns=["input_ids","attention_mask","label"])


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/22500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [ ]:
# Load fresh DistilBERT
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)

# Metrics callback
accuracy = evaluate.load("accuracy")
f1       = evaluate.load("f1")
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {
      "accuracy": accuracy.compute(predictions=preds, references=p.label_ids)["accuracy"],
      "f1":       f1.compute(predictions=preds, references=p.label_ids)["f1"],
    }

# TrainingArguments
args = TrainingArguments(
    "imdb-distilbert-run1",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
    run_name=None,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    compute_metrics=compute_metrics,
)

# Train
trainer.train()

# Evaluate on validation + test
val_metrics = trainer.evaluate(ds["validation"])
test_metrics = trainer.evaluate(ds["test"])
print(" Fine-Tuned Validation:", val_metrics)
print(" Fine-Tuned Test      :", test_metrics)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.274400,0.226938,0.915200,0.918020
2,0.170100,0.276980,0.912000,0.912837
3,0.099800,0.334397,0.915200,0.918462


 Fine-Tuned Validation: {'eval_loss': 0.22693827748298645, 'eval_accuracy': 0.9152, 'eval_f1': 0.9180201082753287, 'eval_runtime': 17.2333, 'eval_samples_per_second': 145.068, 'eval_steps_per_second': 4.584, 'epoch': 3.0}
 Fine-Tuned Test      : {'eval_loss': 0.2265179455280304, 'eval_accuracy': 0.909, 'eval_f1': 0.9090109186897573, 'eval_runtime': 174.7157, 'eval_samples_per_second': 143.09, 'eval_steps_per_second': 4.476, 'epoch': 3.0}


In [ ]:
# Save to drive
drive_path = "/content/drive/MyDrive/imdb-distilbert-finetuned"
trainer.save_model(drive_path)          # model weights + config
tokenizer.save_pretrained(drive_path)   # tokenizer files



('/content/drive/MyDrive/imdb-distilbert-finetuned/tokenizer_config.json',
 '/content/drive/MyDrive/imdb-distilbert-finetuned/special_tokens_map.json',
 '/content/drive/MyDrive/imdb-distilbert-finetuned/vocab.txt',
 '/content/drive/MyDrive/imdb-distilbert-finetuned/added_tokens.json',
 '/content/drive/MyDrive/imdb-distilbert-finetuned/tokenizer.json')

In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
trainer.push_to_hub("rxnu/imdb-distilbert-finetuned")

Uploading...:   0%|          | 0.00/268M [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/rxnu/imdb-distilbert-run1/commit/6f0f21cf6c63ddec3afb1526036a71ce4a93f6c7', commit_message='rxnu/imdb-distilbert-finetuned', commit_description='', oid='6f0f21cf6c63ddec3afb1526036a71ce4a93f6c7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/rxnu/imdb-distilbert-run1', endpoint='https://huggingface.co', repo_type='model', repo_id='rxnu/imdb-distilbert-run1'), pr_revision=None, pr_num=None)

### Key Takeaways
- **Strong Performance:** Validation accuracy (91.5 %) and F1 (0.918) indicate the model learned relevant features well.
- **Generalization:** Test accuracy remains high (90.9 %), suggesting minimal overfitting.
- **Initialized Layers:** Newly initialized classifier layers may benefit from additional fine-tuning.
- **Next Steps:**  
  1. Continue fine-tuning on domain-specific reviews so the classifier layers learn more nuanced patterns.  
  2. Tweak the learning-rate schedule or add a couple more epochs to squeeze out a bit more performance.  
  3. Evaluate on a completely unseen, real-world dataset (e.g., recent IMDB reviews) to confirm deployment readiness.  